# AXI test 20: the serving side a seventh time, requests in ordinary words
**Why this test exists.** In the sixth run (test 19) the jobs of 100 steps were held exactly by the grammar that names the coordinates (6,400 of 6,400 steps), against 97.6 % for the capped grammar. In every run so far the request was written with coordinates (`Delete the service at (2,1).`). A user writes a name: *remove the billing service*. This run gives the model the request in ordinary words, with a short list of the names and where they sit. It runs three cells on the same 64 jobs of the sixth run (jobs 192 to 255, 100 steps): `C0` (the request with coordinates, plus the list of names: the control), `W0` (the request in words, the capped grammar: does the model read the name?) and `WN` (the request in words, and the host reads the sentence by fixed rules and builds the named grammar; when the rules cannot read a sentence the capped grammar is used). There is **no retry in any cell**. The predictions and the reading rules were written before this run in `docs/PROVIDER_SIDE_FREE_TEST.md`.

Model: **Qwen2.5-7B-Instruct, 4-bit AWQ**, engine **vLLM 0.31.0**, cache fixed at **80,000 tokens**, GPU **T4**.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. **Choose the T4 even if your plan offers a faster GPU:** every earlier number is from a T4, and another GPU would make the comparison with them unreliable. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. **Keep this browser tab open** while a cell runs.

**How long it takes (a guess from the earlier runs):** installing the engine about 5 minutes, loading the model 5 to 10 minutes, the pre-flight check and the safety check 3 minutes, the three cells about 33 minutes. In all about 55 to 70 minutes.

**If Colab stops** (the tab closes, the GPU is taken away): reconnect, run Cells 1 to 6 again (do not forget Cell 4b), and then run the cell you were in. Finished cells are skipped; a cell that was cut off starts again from its beginning (the longest takes about 13 minutes).

Do not paste any keys anywhere.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: get the code from GitHub. If it stops with 'the code is not on GitHub yet', finish the update 69 commands in Codespaces (git push) and run this cell again

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_free.py"), "the code is not on GitHub yet: finish the update 69 commands (git push) first"
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_long.py"), "the code is not on GitHub yet: finish the update 69 commands (git push) first"
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_typed.py"), "the code is not on GitHub yet: finish the update 69 commands (git push) first"
assert os.path.exists("/content/axi/python/axi/experiments/q_measure.py"), "the code is not on GitHub yet: finish the update 69 commands (git push) first"
print("code ok")

### Cell 3: install the engine, version 0.31.0 (about 5 minutes). If Colab says 'Restart session', press it, then run Cells 2, 3, 4 and 4b again (Cell 3 will then be quick)

In [ ]:
!pip -q install vllm==0.31.0

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). If it prints 'Drive NOT connected', the run still works but nothing survives a disconnect

In [ ]:
import sys, os, json, time, glob, re
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_provider_free")

### Cell 4b: fix the output stream (about 1 second). vLLM 0.31.0 asks the notebook for a file number of its output and a Colab cell has none, so Cell 6 stops with `io.UnsupportedOperation: fileno`. This cell gives the notebook's output the usual numbers 1 and 2. Run it once after every restart, before Cell 6

In [ ]:
import sys
for name, fd in (("stdout", 1), ("stderr", 2)):
    s = getattr(sys, name)
    try: s.fileno()
    except Exception: s.fileno = (lambda fd=fd: fd)
print("output numbers:", sys.stdout.fileno(), sys.stderr.fileno())

### Cell 5: PRE-FLIGHT CHECK (about 10 seconds). Run it BEFORE Cell 6, never after: once the model is loaded the GPU is in use and the check will say so, which is not a fault. It checks that the jobs are the sixth run's, that every job has 100 steps, that the files the test imports are the pinned ones, that the expressions are the ones on the page, that the oracle's reply is inside every expression, that the first request of each cell is built, and that the GPU is a T4 with nothing running on it. **If it says RED, do not run the next cells: send me the printout**

In [ ]:
from axi.experiments import provider_side_free as F
res = F.preflight()
green = F.print_preflight(res)
assert green, "PRE-FLIGHT IS RED: do not run the next cells. Send me the printout above."

### Cell 6: load the model and the engine with one expression per request (downloads about 5.5 GB once; about 5 to 10 minutes). It prints the engine's own numbers. If it fails with a red error, stop and send me the red text (if the red text says `fileno`, run Cell 4b first; if it says `StructuredOutputsParams`, send it to me and do nothing else)

In [ ]:
from axi.experiments import provider_side_typed as T
from axi.experiments import provider_side_long as L
from axi.experiments import provider_side_free as F
from axi.experiments import q_measure as Q
eng = T.TypedVLLMEngine()
info = eng.info(); print(json.dumps({k: v for k, v in info.items() if k != "constraint"}, indent=1))
print("constraint:", info["constraint"]["kind"], "base expression with a name of at most", info["constraint"]["name_cap"], "characters", info["constraint"]["regex_sha256"][:16], "(each request carries its own expression)")
eng.warm_up()
print("cache size the engine will use:", info["capacity_tokens"], "tokens (", info["capacity_source"], ")")

def go(label):
    """Runs one cell unless its file is already there; saves it and copies it to Drive. A cell that was cut off starts again."""
    name = f"axi_provider_f_{label}.json"; K, first, arm, mode, shape = F.CELLS[label]
    print(f"=== {label}: jobs {first} to {first + K - 1}, request {mode}, grammar {shape}: {store.restore(name)}")
    if os.path.exists(store.path(name)): print("   already finished, skipping"); return
    t0 = time.time()
    def progress(i): print(f"   round {i}   {time.time() - t0:6.0f} s", end="\r")
    c = F.run_cell(eng, label, progress=progress)
    F.save_cell(c, store.path(name)); store.persist(name)
    s = F.summarize_cell(c); f = lambda x, d=1: "-" if x is None else f"{x:.{d}f}"
    print(f"\n   done: {s['steps_done']} steps, {s['wall_seconds']:.0f} s, {f(s['seconds_per_step'], 3)} s/step, exact steps {s['exact_share'] * 100:.1f} %")

### Cell 7: SAFETY CHECK (about 2 minutes). 3 jobs of 3 steps of each of the three cells. It prints what the engine answers. **A wrong answer is not a STOP** (that is what the test is for). If it says STOP, do not run the next cells: send me the printout

In [ ]:
smoke = {lab: L.run_sessions(eng, [F.FreeSession(L.SEED * 1000 + e, 3, F.CELLS[lab][3], F.CELLS[lab][4]) for e in (192, 193, 194)]) for lab in F.ORDER}
for name, sm in smoke.items():
    print(name)
    for x in sm["sessions"][0]["log"]: print(f"  step {x['t']}: prompt {x['n_prompt']:5d}  cached {x['n_cached']}  new {x['n_new']:3d}  {x['text'][:50]!r}   request: {x['request'][:60]!r}")
allrep = [(x["parsed"], x["n_cached"], x["in_language"], x["n_new"]) for sm in smoke.values() for s in sm["sessions"] for x in s["log"]]
ok_cached = all(c is not None for _, c, _, _ in allrep)
ok_parse = all(p for p, _, _, _ in allrep)
ok_lang = all(l for _, _, l, _ in allrep)
ok_long = all(n < 96 for _, _, _, n in allrep)
print()
print("STOP: this vLLM version does not report cached tokens. Send me the printout and the version above." if not ok_cached else
      "STOP: a reply of the safety cell was refused by the strict parser, so the constraint is not working. Send me the printout." if not ok_parse else
      "STOP: a reply of the safety cell is outside the expression of its own request, so one expression per request is not working. Send me the printout." if not ok_lang else
      "STOP: a reply of the safety cell used all 96 new tokens. Send me the printout." if not ok_long else
      "OK: cached counts are reported, every reply is well formed and inside the expression of its request, and the replies end. You can go on.")

### Cell 8: the three cells, in this order: `C0` (request with coordinates), `W0` (request in words, capped grammar) and `WN` (request in words, the host reads it and builds the named grammar). Jobs 192 to 255, 100 steps each. About 33 minutes in all. A counter shows the round; the longest cell takes about 13 minutes. Do not close the tab

In [ ]:
for label in F.ORDER: go(label)

### Cell 9: put the finished cells together, print the table, the split by wording pool, the first misses, the 16 reading-rule lines and Q = O / M of every cell, and download your file. You can run it at any time, with whatever cells are finished

In [ ]:
from google.colab import files
paths = sorted(glob.glob(store.path("axi_provider_f_*.json")))
print(len(paths), "cells:", [os.path.basename(p) for p in paths])
m = F.merge_files(paths, store.path("axi_provider_free.json")); store.persist("axi_provider_free.json")
S = F.summarize_all(m["cells"]); F.print_cells(S); F.print_split(S); F.print_misses(S); F.print_verdict(F.verdict_free(m["cells"]))
print(); print(Q.render_table({"Q = O / M of the cells of this run": [Q.q_row(c) for c in m["cells"]]}))
files.download(store.path("axi_provider_free.json"))

### Done
File downloaded: `axi_provider_free.json`. Upload it to the chat, with the cell files from your Drive folder `axi_provider_free` (the `axi_provider_f_*.json` files), a screenshot of the printed table, of the split and of the 16 reading-rule lines, the printout of Cell 5 and of Cell 7. Do not paste any keys anywhere.